# Week 6 · SFT 指令微调

> **本周一句话**:让模型从"续写"升级到"按指令生成" —— 给它"题目 + 风格"两个槽位,它返回符合要求的诗。

这是从"语言模型"到"指令模型"的关键一步。技术上需要:特殊 token、prompt mask、vocab 扩展时的权重迁移、小 lr 微调。

## 0. 本周目标

| 维度 | v0.8(预训练) | v0.9(SFT) |
|---|---|---|
| 用法 | `generate("月", 200)` 续写 | `prompt = 题目=春雨 + 风格=五绝` 按指令生成 |
| 数据 | 22M 字符自由文本 | 37k 条结构化 (题目, 风格, 正文) |
| vocab | 14,477 | 14,482(加 5 个特殊 token) |
| lr | 3e-4(预训练) | 5e-5(小 6 倍) |
| val loss | 4.03 | ~3.55(注意 SFT loss << 预训,因为 prompt 被 mask) |

能跑的 prompt 例子:
- "请以「春雨」为题,写一首五言绝句"
- "请以「程序员」为题,写一首七言绝句"(看老模型对现代题材的反应)

## 1. 前置知识

**必备**:
- Week 5 跑完(v0.8 best 存在)
- 知道什么是"loss mask"

**这周第一次遇到**:
- 特殊 token(`<|题目|>` 等)的设计与编码
- prompt vs completion 的概念
- `cross_entropy(ignore_index=-100)` 用法
- vocab 扩展时迁移权重的边界条件

## 2. 核心概念

### 2.1 SFT 数据格式

每个样本是一段拼接后的文本:

```
<|题目|>静夜思<|风格|>五言绝句<|开始|>床前明月光，疑是地上霜。举头望明月，低头思故乡。<|结束|>
```

前半截(`<|题目|>...<|开始|>`)叫 **prompt**,后半截(诗本身 + `<|结束|>`)叫 **completion**。

**这一段一次性整个塞进模型 forward**,模型在每个位置都要预测下一个 token。但 —— 

### 2.2 prompt mask:只在 completion 上算 loss

问题:如果对整段算 loss,模型会努力学"如何续写题目和风格" —— 那是我们给它的输入,本就不该学。

**修复**:把 prompt 部分的 target 设成 `-100`,`F.cross_entropy(ignore_index=-100)` 自动跳过。

```python
L = len(ids) - 1
x[i, :L] = torch.tensor(ids[:-1])    # 输入
y[i, :L] = torch.tensor(ids[1:])     # 目标

start_pos = ids.index(START_ID)       # <|开始|> 的位置
y[i, :start_pos] = -100               # ← prompt 部分 mask 掉
```

**注意 mask 的索引**:y[i, :start_pos] 而不是 :start_pos+1。因为 y[t] 预测的是 ids[t+1],我们想从 ids[start_pos+1](正文第一个字)开始算 loss,对应 y[start_pos:] 保留。

### 2.3 特殊 token 必须接在 vocab 末尾

v3 tokenizer 的设计:

```
v2: id  0..14476  (字符 + 标点)
v3: id  0..14476  (完全不动!)
         14477   <|题目|>
         14478   <|风格|>
         14479   <|开始|>
         14480   <|结束|>
         14481   <|pad|>
```

**为什么必须 append**?Week 6 要把 v0.8 (vocab=14477) 的权重灌进 v0.9 (vocab=14482) —— 只有旧 ID 完全不动,前 14477 行 embedding 才能 1:1 拷贝。如果在中间插入,所有 ID 重排,权重对不上 → 等于从头训练。

**通用原则**:扩 vocab 永远在尾部加,不在中间插。

### 2.4 vocab 扩展的权重迁移

```python
def transfer_weights_to_larger_vocab(new_model, v08_state, old_vocab=14477):
    new_state = new_model.state_dict()
    for key, val in v08_state.items():
        if key in ("token_embedding.weight", "lm_head.weight"):
            new_state[key][:old_vocab] = val   # 前 14477 行拷贝,后 5 行保留随机初始化
        else:
            new_state[key] = val               # 其他层 shape 不变,直接拷贝
    new_model.load_state_dict(new_state)
```

**关键 invariant**:旧字的 embedding 行 / lm_head 行 = v0.8 学到的;新 5 行 = 随机初始化,SFT 训练时会被快速学好(因为 `<|题目|>` 等出现频次极高,梯度信号强)。

**结果**:v0.9 一开始就具备 v0.8 的全部语言能力,只是多了 5 个新词需要学。比从头训 v0.9 快 100×。

### 2.5 为什么 SFT lr = 预训练 lr / 6

v0.8 预训用 max_lr=3e-4;v0.9 SFT 用 5e-5。

**原因**:
- v0.8 已经训了 10000 步,权重在一个相对好的位置
- 大 lr 会"踢飞"已学到的语言能力,模型只剩下"按指令生成"但不会写诗了 —— 灾难性遗忘
- 小 lr 让模型"在已有能力之上做小修改"

通用规律:**任何 fine-tune 阶段的 lr 都比预训阶段小 5-10×**。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `data/classify.py` | 35 | 按句数/字数判定 五绝/七绝/五律/七律 |
| `tokenizer/build_v3.py` | 35 | v2 + 5 个特殊 token → v3 |
| `data/prepare_sft.py` | 60 | 用 classify + v3 编码,构造 list[list[int]] |
| `configs/config.py:SFTCfg` | 9 | 3000 步、lr 5e-5、batch 32、max_len 128 |
| `train/train_v09_sft.py` | 165 | vocab 扩展 + 权重迁移 + prompt mask + 训练 |
| `inference/generate_sft.py` | 105 | CLI 生成,带 `--battery` 12 题套件 |

## 4. 动手做

In [ ]:
import subprocess, sys
from pathlib import Path


def _find_repo_root() -> Path:
    """定位仓库根目录(含 pyproject.toml + train/),不依赖 notebook 工作目录。
    - 本地/Colab:cwd 在仓库内,从 cwd 往上找即命中。
    - 魔搭 ModelScope:kernel cwd 在 /mnt/workspace,但仓库克隆在 home,
      所以再去 home / 常见根目录下搜几层。"""
    def ok(d):
        return (d / "pyproject.toml").is_file() and (d / "train").is_dir()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if ok(d):
            return d
    bases, seen = [here, Path.home(), Path("/home"), Path("/root"), Path("/mnt")], set()
    for base in bases:
        try:
            base = base.resolve()
        except Exception:
            continue
        if not base.is_dir() or base in seen:
            continue
        seen.add(base)
        for depth in ("*", "*/*", "*/*/*"):
            for f in base.glob(f"{depth}/pyproject.toml"):
                if (f.parent / "train").is_dir():
                    return f.parent.resolve()
    raise RuntimeError(f"找不到仓库根(应含 pyproject.toml + train/),cwd={here},home={Path.home()}")


REPO = _find_repo_root()                      # 项目根的绝对路径
print(f"REPO = {REPO}")


def run(cmd):
    """跑子进程并把输出实时打印到 cell。
    - "python" 换成 sys.executable,确保用当前 kernel 解释器。
    - 形如 "../train/x.py" 的相对路径统一解析成 REPO 下的绝对路径,
      不再依赖 notebook 的工作目录(魔搭与本地/Colab 的 cwd 不一致)。
    - cwd=REPO 兜底:即便脚本内部用了相对路径也能找到文件。
    - Popen 逐行回读才能在 cell 里实时看到脚本的 print。"""
    cmd = [sys.executable if c == "python" else c for c in cmd]
    cmd = [str(REPO / c.removeprefix("../")) if isinstance(c, str) and c.startswith("../") else c
           for c in cmd]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1, cwd=str(REPO),
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"{cmd} 退出码 {proc.returncode}")

# 1. 建 v3 tokenizer(秒级)
run(["python", "../tokenizer/build_v3.py"])
# 预期: old vocab 14477 -> new vocab 14482

In [ ]:
# 2. 构造 SFT 样本(~30 秒)
run(["python", "../data/prepare_sft.py"])
# 预期: ~37000 条样本,长度中位数 ~50,最长 ~100

In [ ]:
# 3. SFT 训练(~10 分钟 T4)
run(["python", "../train/train_v09_sft.py"])
# 预期:
#   开始 val ~5-6(刚加了 5 个新 token,新权重还没学好)
#   500 步内降到 ~4
#   3000 步收敛到 ~3.55
#   注意: SFT loss 远低于预训 loss(因为 prompt 部分被 mask)

In [ ]:
# 4. 写诗
run(["python", "../inference/generate_sft.py",
                "--title", "春雨", "--style", "五言绝句"])

In [ ]:
# 5. 12 题套件(对照不同难度题目的发挥)
run(["python", "../inference/generate_sft.py", "--battery"])
# 4 档 × 3 题:
#   A 档 古典正统(明月、江雪、饮酒) — 应该最稳
#   B 档 古代少见(观棋、夜读、卖花) — 句式应该 OK
#   C 档 现代题用古意(咖啡、地铁、独居) — 看模型有没有"硬翻"能力
#   D 档 完全现代(程序员、深度学习、996) — 大概率胡来,但很有趣

## 5. 自测题

**A. 数据格式**
- A1 SFT 样本里 `<|结束|>` 必须有吗?不加会怎样?
- A2 题目 + 风格 + 正文的顺序能换吗?换成 风格 + 题目 + 正文 训出来会不同吗?
- A3 我们 max_len=128,SFT 数据平均长度 ~50。剩余 ~78 个位置 pad,这些位置算 loss 吗?

**B. prompt mask**
- B1 如果不 mask prompt(整段算 loss),模型最终会变成什么样?
- B2 mask 用 `-100` 而不是 `0`,原因?
- B3 我们的 mask 用 `y[i, :start_pos] = -100`,如果改成 `y[i, :start_pos+1] = -100` 会怎样?

**C. vocab 扩展**
- C1 如果不迁移 v0.8 权重,从头训 v0.9,要训多少步才能达到 val 3.55?(粗估)
- C2 5 个新 token 的 embedding 初始化为 0 vs 随机正态,有什么区别?
- C3 如果旧 ID 顺序变了(比如新加的 token 插在中间),迁移代码会失败还是静默错乱?

**D. 超参**
- D1 lr 从 5e-5 改成 3e-4(同预训),前 100 步会观察到什么现象?
- D2 SFT 训 3000 步够用吗?训 10000 会怎样?
- D3 batch_size 32 vs 16 在 SFT 阶段哪个更"稳"?为什么?

## 6. 容易踩的坑

**坑 1:`stoi["<|题目|>"]` 编不出来 —— 因为传成了 "<|题目|>" 7 个字符**

special token 必须**整体**当一个 key 去查,我们的 `encode_pieces` 函数会先检测整段是不是 special token,是就 `stoi[piece]`,否则才逐字。

**坑 2:Dropout 在 SFT 里开太大模型背不住样本**

v0.8 配的 dropout=0.1 在预训刚好,SFT 阶段样本只有 37k 比预训少 50×,dropout 太大会让模型记不住格式。可以试 dropout=0.05 看看 val 是不是更低。

**坑 3:pad_id 选了一个数据里出现过的 ID → 模型把 pad 当真信号学**

我们用 `<|pad|>` 这种独立 token,id=14481,数据里绝不会自然出现。**绝对不能**拿 0 (某个汉字的 ID) 当 pad。

**坑 4:迁移权重时手抖把 `key in (...)` 改成 `key == ...` 漏了 lm_head**

lm_head 和 token_embedding 都需要前 14477 行特殊处理 —— 这两个都是 `(vocab_size, n_embed)` shape。漏一个 → 新 token 的 logits 永远 garbage。

**坑 5:generate 时 prompt 编码忘了加 `<|开始|>`**

prompt 必须以 `<|开始|>` 结尾,模型才知道"现在该输出正文了"。漏了 → 模型继续生成"风格"槽位的内容,胡言乱语。

## 7. 进入 Week 7 前

现在你应该:
- ☑ `tokenizer_v3.pkl` 和 `sft_data.pt` 都存在
- ☑ `checkpoints/v09_sft/best.pt` 存在
- ☑ `--battery` 跑出来 A 档(古典)能写出 4 句押韵的诗,D 档(996)很搞笑
- ☑ 能解释 prompt mask 的"为什么 -100 而不是 0"

Week 7 我们要在 SFT 之上再加一层 **DPO 偏好对齐**:用规则评分器(押韵 / 句数 / 题目相关)自动给候选打分,代替昂贵的人工 RLHF。这是 SFT → "更对齐人类偏好"的关键一步。